# Engram: does more memory text help?

The four networks at 1,500 tokens, against their saved 800-token answers. Development conversations 0 to 4. Benchmark data only.

**Kaggle:** Settings > Accelerator > GPU T4 x2, and Settings > Turn on internet. Then Run > Run all. Keep the tab open.

In [ ]:
COMMIT = "aa6aaf0"
import os, subprocess, time, urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu: raise SystemExit("No GPU. Choose Settings > Accelerator > GPU T4 x2.")
print("GPU:", gpu.replace(chr(10), ", "))
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise SystemExit("No internet. Choose Settings > Turn on internet, then run again.")
WORK = "/kaggle/working"
os.chdir(WORK)
subprocess.run("rm -rf Engram; git clone -q https://github.com/manjunathhari/Engram.git", shell=True, check=True)
os.chdir(f"{WORK}/Engram")
subprocess.run(f"git checkout -q {COMMIT}; git clone -q --depth 1 https://github.com/snap-research/locomo.git", shell=True, check=True)
subprocess.run("pip install -q wordllama scikit-learn spacy; python -m spacy download en_core_web_sm", shell=True, check=True, capture_output=True)
import wordllama, shutil
p = os.path.dirname(wordllama.__file__)
a, b = os.path.join(p, "tokenizers"), os.path.join(p, "tokenizer")
if os.path.isdir(a) and not os.path.exists(b): shutil.copytree(a, b)
subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(5)
MODEL = "hf.co/bartowski/google_gemma-3-4b-it-GGUF:Q4_K_M"
r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
if r.returncode: raise SystemExit("Model download failed: " + r.stderr[-200:])
subprocess.run(["ollama", "cp", MODEL, "gemma3:4b"])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
print("setup ready")

In [ ]:
%%writefile /kaggle/working/token_test.py
"""Does more memory text improve answers? The four networks at 1,500 tokens, against their saved 800-token answers.
Development conversations 0-4 only. Benchmark data only. Same questions, networks and answer prompt as the helper test."""
import argparse, json, os, sys, tempfile, time
import numpy as np
sys.path.insert(0, os.getcwd())
import vps_kit as K
import engram_chat as E
from engram import Config
import engram_parallel as P

ap = argparse.ArgumentParser()
ap.add_argument("--url", required=True); ap.add_argument("--model", required=True)
ap.add_argument("--out", required=True); ap.add_argument("--baseline", required=True)
ap.add_argument("--sample", type=int, default=240); ap.add_argument("--budget", type=int, default=1500)
a = ap.parse_args()
ids = [0, 1, 2, 3, 4]
tools = P._load_tools()
cfg = Config(**json.load(open("final_config.json")))
tmp = tempfile.mkdtemp()
convs = {c: K.load_conv(K.DATA[c]) for c in ids}


def replay(conv, name):
    m = E.ChatMemory(f"{tmp}/{name}.db", cfg=cfg, durable=False, search=P.ChannelSearch("plain", tools=tools))
    last = None
    for t in conv.turns:
        if t["session"] != last:
            if last is not None: m.close_chat(write_note=False, sleep=False)
            m.open_chat(t["date"]); last = t["session"]
        m.add(t["speaker"], t["text"], date=t["date"], ref=t["id"], caption=t["cap"], commit=False)
    m.db.commit(); m.close_chat(write_note=False)
    return m


print("building memories", flush=True)
MEM = {c: replay(convs[c], f"s{c}") for c in ids}
REF = {c: dict(MEM[c].db.execute("SELECT ref, sid FROM sentences").fetchall()) for c in ids}
GOLD = {c: [(q, {REF[c][e] for e in q["ev"] if e in REF[c]}) for q in convs[c].qs if q["cat"] != 5 and q["ev"]] for c in ids}
print("training networks", flush=True)
NET = {c: [P.train([(MEM[d].search, q["q"], g) for d in ids if d != c for q, g in GOLD[d] if g], s) for s in (1, 2, 3)] for c in ids}


def order(c, q):
    s = MEM[c].search
    f = s.features(q)
    p = np.mean([m[k][1].predict_proba(m[k][0].transform(f["tables"][k]))[:, 1] for m in NET[c] for k in P.CHANNELS], axis=0)
    first = f["pool"][np.argsort(-p)].tolist()
    have = set(first)
    return [s.sids[i] for i in first + [x for x in np.argsort(-f["plain"]).tolist() if x not in have]]


def select(c, sids, budget):
    m, out, seen, used = MEM[c], [], set(), 0

    def take(sid):
        nonlocal used
        if sid in seen: return
        seen.add(sid); out.append(sid); used += m._tokens(sid) + 8
    for sid in sids:
        if used >= budget: break
        take(sid)
        nxt = m._next_in_chat(sid)
        if nxt is not None and used < budget: take(nxt)
    return out, used, m.text_of(sorted(out))


plan = [(c, i) for c, i in K.balanced(convs, ids, a.sample) if convs[c].qs[i]["cat"] != 5]
plan = [(c, i) for c, i in plan if any(e in REF[c] for e in convs[c].qs[i]["ev"])]
base = {(r["conv"], r["i"]): r for r in map(json.loads, open(a.baseline)) if r["system"] == "network"}
print(len(plan), "questions;", len(base), "saved 800-token answers", flush=True)
llm = K.LLM(a.url, a.model, "local", 1200)
done = {(r["conv"], r["i"]) for r in map(json.loads, open(a.out))} if os.path.exists(a.out) else set()

# Check first: the same search and model must reproduce the saved run, or the comparison would not be fair.
same_ev, same_ans, checks = 0, 0, 0
for c, i in plan[:20]:
    q = convs[c].qs[i]
    gold = {REF[c][e] for e in q["ev"] if e in REF[c]}
    sids, _, txt = select(c, order(c, q["q"]), 800)
    rec = len(set(sids) & gold) / len(gold)
    pred = llm(K.ANSWER.format(ctx=txt or "(no excerpts)", q=q["q"]), max_tokens=60)[0]
    b = base.get((c, i))
    if b:
        checks += 1
        same_ev += abs(rec - b["recall"]) < 1e-9
        same_ans += pred.strip() == b["pred"].strip()
print(f"reproduction check: evidence same on {same_ev}/{checks}, answers word for word on {same_ans}/{checks}", flush=True)
if checks and same_ev < checks * .7:
    print("STOPPED: search does not reproduce the saved run. Send this to Claude.", flush=True)
    sys.exit(3)

with open(a.out, "a") as out:
    for n, (c, i) in enumerate(plan, 1):
        if (c, i) in done: continue
        q = convs[c].qs[i]
        gold = {REF[c][e] for e in q["ev"] if e in REF[c]}
        sids, used, txt = select(c, order(c, q["q"]), a.budget)
        s800 = select(c, order(c, q["q"]), 800)[0]
        pred, _, secs = llm(K.ANSWER.format(ctx=txt or "(no excerpts)", q=q["q"]), max_tokens=60)
        out.write(json.dumps(dict(conv=c, i=i, system=f"network{a.budget}", cat=K.CATS[q["cat"]], q=q["q"], gold=q["answer"], pred=pred, recall=len(set(sids) & gold) / len(gold), recall800=len(set(s800) & gold) / len(gold), tokens=used, secs=secs)) + "\n")
        out.flush()
        if n % 10 == 0: print(f"question {n}/{len(plan)}", flush=True)
print("Done", flush=True)

In [ ]:
import os, subprocess, sys, time
WORK, SAVE = "/kaggle/working", "/kaggle/working/engram_results"
os.makedirs(SAVE, exist_ok=True); os.chdir(f"{WORK}/Engram")
LOG, OUT = f"{SAVE}/token_run.log", f"{SAVE}/tokens_dev.jsonl"
if subprocess.run(["pgrep", "-f", "token_test.py"], capture_output=True).returncode != 0:
    subprocess.Popen([sys.executable, "-u", f"{WORK}/token_test.py", "--url", "http://localhost:11434/v1", "--model", "gemma3:4b", "--out", OUT, "--baseline", "results/helper_hf_dev.jsonl"], stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    print("test started")
for _ in range(240):
    time.sleep(15)
    t = open(LOG).read().splitlines()
    print(chr(13) + " | ".join(t[-2:])[:150], end="")
    if t and (t[-1].startswith("Done") or "STOPPED" in t[-1] or "Traceback" in "\n".join(t[-30:])): break
print(); print("\n".join(open(LOG).read().splitlines()[-25:])); print(sorted(os.listdir(SAVE)))

When it says Done, send Claude `tokens_dev.jsonl` and `token_run.log` from the `engram_results` folder in the Output panel.